# Gilbert Owen

# 2802432220

# DTSC6008001 – Text Mining

# 3. Classification

### A. Objective

**Input**: `data/processed/articles_preprocessed.parquet` from Task 2 (6,110 articles, 5 labels), already split 70/15/15 into train, validation and test.

**Task**: classify each article into one of the 5 labels with two models, then compare them.

| Model | Family | Reads |
|---|---|---|
| BiLSTM | deep learning | `text_dl`, turned into word ids with a vocabulary built on train only |
| IndoBERT | BERT family | `text_bert`, cut by its own subword tokenizer, at most 512 tokens |

**Rules from the brief**: tune at least 2 hyperparameters for each model, scored on validation. The test set is used once per model, for the final score. Then compare the two models and explain why one does better.

**Target**: Determine which model performs the best on text classification.

### B. Import Libraries and Environment Check

The `deep-learning` environment already has TensorFlow 2.10 with GPU support (RTX 3060, 6 GB), so both models are built in TensorFlow: the BiLSTM with Keras, and IndoBERT through Hugging Face `transformers`. `transformers` is imported in the IndoBERT section, this cell only checks that it is installed.

In [1]:
import random
from importlib.util import find_spec
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

for gpu in tf.config.list_physical_devices("GPU"): 
    tf.config.experimental.set_memory_growth(gpu, True)

print("TensorFlow: {}".format(tf.__version__))
print("GPU: {}".format(tf.config.list_physical_devices("GPU")))
print("transformers installed: {}".format(find_spec("transformers") is not None))

TensorFlow: 2.10.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
transformers installed: True


### C. Configuration

All settings live in this one cell. `SEED` is the same as in Task 2, and it is set for Python, NumPy and TensorFlow, so weight initialisation and shuffling come out the same on every run.

In [3]:
SEED = 42
VOCAB_SIZE = 20000
MAX_LEN = 512
MAX_LEN_BERT = 512   # IndoBERT length, counted in subwords (chosen in I1)

PROC_DIR = Path("../data/processed")
PREPROCESSED_PARQUET = PROC_DIR / "articles_preprocessed.parquet"

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("articles_preprocessed.parquet found: {}".format(PREPROCESSED_PARQUET.exists()))

articles_preprocessed.parquet found: True


### D. Load the Preprocessed Articles

Read the file Task 2 saved in Section J and cut it into the three parts by its `split` column. The split is not redone here, so every model is scored on the same 917 test articles. Expected: 6,110 articles, train 4,277, val 916, test 917.

In [4]:
df = pd.read_parquet(PREPROCESSED_PARQUET)

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print("Articles: {} | train {} | val {} | test {}".format(len(df), len(train_df), len(val_df), len(test_df)))
pd.crosstab(df["label"], df["split"], margins=True)

Articles: 6110 | train 4277 | val 916 | test 917


split,test,train,val,All
label,,,,
business,183,855,183,1221
entertainment,163,761,163,1087
health,192,895,192,1279
sports,192,895,192,1279
technology,187,871,186,1244
All,917,4277,916,6110


### E. Label Ids

Both models output a number for each label, not the label's name. The labels are sorted, so `business` is always 0 and the mapping is the same on every run. Both models use these same `y` arrays.

In [5]:
LABELS = sorted(df["label"].unique())
label_to_id = {label: i for i, label in enumerate(LABELS)}

y_train = train_df["label"].map(label_to_id).to_numpy()
y_val = val_df["label"].map(label_to_id).to_numpy()
y_test = test_df["label"].map(label_to_id).to_numpy()

print(label_to_id)
print("y_train: {} | y_val: {} | y_test: {}".format(y_train.shape, y_val.shape, y_test.shape))

{'business': 0, 'entertainment': 1, 'health': 2, 'sports': 3, 'technology': 4}
y_train: (4277,) | y_val: (916,) | y_test: (917,)


### F. BiLSTM Input

The BiLSTM reads a sequence of word ids, and every sequence in a batch must have the same length. That needs two settings:

| Setting | What it does |
|---|---|
| `VOCAB_SIZE` | only the most common words get their own id. Every other word becomes one shared "unknown" id. Id 0 is kept for padding. |
| `MAX_LEN` | shorter articles are filled up with 0s, and longer ones are cut, keeping the start of the article |

The vocabulary is built on **train only**. If val or test words were in it, the model would get ids for words it is never trained on, and the val and test scores would no longer show how it handles new articles.

#### F1. Vocabulary Size

How many distinct words train has, and what share of all train tokens the most common words cover. The last line shows how many val tokens are missing from the train vocabulary: those are the words a model built on train will see as unknown.

In [5]:
from collections import Counter

train_vocab = Counter(word for text in train_df["text_dl"] for word in text.split())
total = sum(train_vocab.values())
counts = [count for _, count in train_vocab.most_common()]

print("Distinct words in train: {}".format(len(train_vocab)))
print("Seen at least 2 times: {} | at least 5 times: {}".format(sum(c >= 2 for c in counts), sum(c >= 5 for c in counts)))

for size in [10000, 20000, 30000, 50000]:
    print("Top {}: {:.1%} of train tokens".format(size, sum(counts[:size]) / total))

val_tokens = [word for text in val_df["text_dl"] for word in text.split()]
unseen = sum(word not in train_vocab for word in val_tokens) / len(val_tokens)
print("Val tokens not in train vocabulary: {:.1%}".format(unseen))

Distinct words in train: 49826
Seen at least 2 times: 29643 | at least 5 times: 16700
Top 10000: 93.0% of train tokens
Top 20000: 97.1% of train tokens
Top 30000: 98.6% of train tokens
Top 50000: 100.0% of train tokens
Val tokens not in train vocabulary: 2.4%


#### F2. Sequence Length

For each candidate `MAX_LEN`: the share of train articles that fit whole, and the share of all train tokens that are kept after cutting.

In [6]:
for max_len in [256, 384, 512, 640]:

    fit = (train_df["n_tokens"] <= max_len).mean()
    kept = np.minimum(train_df["n_tokens"], max_len).sum() / train_df["n_tokens"].sum()
    print("MAX_LEN {}: {:.1%} of articles fit whole | {:.1%} of tokens kept".format(max_len, fit, kept))

MAX_LEN 256: 31.0% of articles fit whole | 67.6% of tokens kept
MAX_LEN 384: 68.0% of articles fit whole | 86.1% of tokens kept
MAX_LEN 512: 87.7% of articles fit whole | 94.0% of tokens kept
MAX_LEN 640: 95.2% of articles fit whole | 97.0% of tokens kept


**What the numbers show**: Each larger vocabulary covers more of the train tokens, but the gain gets smaller: 10,000 words cover 93.0%, 20,000 cover 97.1%, and 30,000 only reach 98.6%. Each longer `MAX_LEN` also fits more articles whole and keeps more tokens, with the same shrinking gain: from 384 to 512 it keeps 7.9% more tokens (86.1% → 94.0%), but from 512 to 640 only 3.0% more (94.0% → 97.0%).

**Decision** (`VOCAB_SIZE` and `MAX_LEN`): `VOCAB_SIZE` = 20,000 and `MAX_LEN` = 512.

**Why**: We cap the vocabulary at 20,000 words, which covers 97.1% of the train tokens. The next 10,000 words only add 1.5% (98.6% at 30,000), which is small compared to the extra complexity and model size they cost. For the sequence length we choose 512, which keeps 94.0% of the train tokens. Going up to 640 only adds 3.0% more, but makes the model heavier and slower. 512 is also the same length as IndoBERT's limit, although IndoBERT counts subwords rather than words, so it will see fewer words of each article than the BiLSTM.

### G. Text to Word Ids

`TextVectorization` builds the vocabulary from train `text_dl` only (`adapt`), then turns every article into `MAX_LEN` word ids. `text_dl` was already lowercased and tokenized in Task 2, so it only needs to be split on spaces (`standardize=None`). Ids 0 (padding) and 1 (unknown) are counted inside `VOCAB_SIZE`, and long articles are cut at the end, so the start is kept.

In [6]:
vectorizer = tf.keras.layers.TextVectorization(max_tokens=VOCAB_SIZE, standardize=None, split="whitespace", output_sequence_length=MAX_LEN)
vectorizer.adapt(train_df["text_dl"].to_numpy())

X_train = vectorizer(train_df["text_dl"].to_numpy()).numpy()
X_val = vectorizer(val_df["text_dl"].to_numpy()).numpy()
X_test = vectorizer(test_df["text_dl"].to_numpy()).numpy()

vocab = vectorizer.get_vocabulary()
print("Vocabulary: {} | first 10: {}".format(len(vocab), vocab[:10]))
print("X_train: {} | X_val: {} | X_test: {}".format(X_train.shape, X_val.shape, X_test.shape))

Vocabulary: 20000 | first 10: ['', '[UNK]', 'yang', 'dan', 'di', 'ini', 'dengan', 'untuk', 'dari', 'dalam']
X_train: (4277, 512) | X_val: (916, 512) | X_test: (917, 512)


Check one article: its first words next to their ids, and the share of val word ids that are unknown (id 1).

In [7]:
print(train_df.loc[0, "text_dl"].split()[:10])
print(X_train[0][:10])
print([vocab[i] for i in X_train[0][:10]])
print("Unknown share in val: {:.1%}".format((X_val == 1).sum() / (X_val > 0).sum()))

['harga', 'emas', 'antam', 'hari', 'ini', 'tidak', 'mengalami', 'perubahan', 'alias', 'stagnan']
[  82  261 1469   62    5   13  183  318 1616 4408]
['harga', 'emas', 'antam', 'hari', 'ini', 'tidak', 'mengalami', 'perubahan', 'alias', 'stagnan']
Unknown share in val: 4.7%


### H. BiLSTM

#### H1. Model

| Layer | What it does |
|---|---|
| `Embedding` | turns each word id into a vector of `embed_dim` numbers, learned during training. `mask_zero=True` makes the next layer skip the padding 0s |
| `Bidirectional(LSTM)` | one LSTM reads the article left to right, another right to left, and their last states are joined, so each side sees the full article |
| `Dropout` | randomly switches off part of the values during training, so the model cannot rely on a few words |
| `Dense` (softmax) | gives one probability per label |

The settings are arguments of `build_bilstm`, so the tuning step can change them without rewriting the model.

In [8]:
def build_bilstm(embed_dim=128, units=64, dropout=0.3, learning_rate=1e-3):
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(VOCAB_SIZE, embed_dim, mask_zero=True),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(units)),
        tf.keras.layers.Dropout(dropout),
        tf.keras.layers.Dense(len(LABELS), activation="softmax"),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

model = build_bilstm()
model.build(input_shape=(None, MAX_LEN))
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, None, 128)         2560000   
                                                                 
 bidirectional (Bidirectiona  (None, 128)              98816     
 l)                                                              
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense (Dense)               (None, 5)                 645       
                                                                 
Total params: 2,659,461
Trainable params: 2,659,461
Non-trainable params: 0
_________________________________________________________________


#### H2. Baseline Run

One run with the default settings before any tuning, to check that the model learns at all. It trains on train, and early stopping watches the val loss: training stops when the val loss has not improved for 2 epochs, and the best weights are kept. The score is on val, with accuracy and macro-F1 (every label weighted equally). Test is not touched.

In [9]:
from sklearn.metrics import accuracy_score, f1_score

early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

model = build_bilstm()
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10, batch_size=32, callbacks=[early_stop], verbose=2)

val_pred = model.predict(X_val, verbose=0).argmax(axis=1)
print("Epochs run: {}".format(len(history.history["loss"])))
print("Val accuracy: {:.3f} | Val macro-F1: {:.3f}".format(accuracy_score(y_val, val_pred), f1_score(y_val, val_pred, average="macro")))


Epoch 1/10
134/134 - 22s - loss: 1.0475 - accuracy: 0.6320 - val_loss: 0.7052 - val_accuracy: 0.7806 - 22s/epoch - 161ms/step
Epoch 2/10
134/134 - 9s - loss: 0.4009 - accuracy: 0.8651 - val_loss: 0.2591 - val_accuracy: 0.9269 - 9s/epoch - 70ms/step
Epoch 3/10
134/134 - 10s - loss: 0.2997 - accuracy: 0.9088 - val_loss: 0.2935 - val_accuracy: 0.9116 - 10s/epoch - 75ms/step
Epoch 4/10
134/134 - 10s - loss: 0.1175 - accuracy: 0.9638 - val_loss: 0.2364 - val_accuracy: 0.9214 - 10s/epoch - 72ms/step
Epoch 5/10
134/134 - 9s - loss: 0.0504 - accuracy: 0.9883 - val_loss: 0.1908 - val_accuracy: 0.9410 - 9s/epoch - 70ms/step
Epoch 6/10
134/134 - 9s - loss: 0.0195 - accuracy: 0.9963 - val_loss: 0.1990 - val_accuracy: 0.9421 - 9s/epoch - 71ms/step
Epoch 7/10
134/134 - 10s - loss: 0.0100 - accuracy: 0.9988 - val_loss: 0.1721 - val_accuracy: 0.9520 - 10s/epoch - 71ms/step
Epoch 8/10
134/134 - 10s - loss: 0.0040 - accuracy: 1.0000 - val_loss: 0.1888 - val_accuracy: 0.9498 - 10s/epoch - 73ms/step
Epoch

**What the baseline shows**: The baseline reaches 1.000 accuracy on the training data at epoch 8 (0.999 at the restored epoch), which suggests overfitting or possible data leakage. Data leakage is unlikely, because we already ran the raw data audit, removed the leakage, and did the leak check again. Training ran for 9 of 10 epochs: the best val_loss was 0.1721 at epoch 7, and because val_loss did not improve over the next 2 epochs, early stopping ended the run and kept the epoch-7 weights. Validation accuracy is about 0.05 lower than training accuracy at that epoch, giving a val accuracy of 0.952 and a macro-F1 of 0.952.

#### H3. Tuning Dropout and Units

We tune `dropout` and `units` because both relate to the overfitting seen in the baseline. Dropout controls how the model learns: during training it randomly switches off a share of the BiLSTM's outputs at each step (30% in the baseline), so the model cannot rely on a single clue. Units set how much the BiLSTM can memorise, so a larger value makes overfitting easier. Every combination is scored on val, the one with the best val macro-F1 wins, and test is not used yet.

Grid: `dropout` in {0.3, 0.5} x `units` in {32, 64, 128} = 6 runs, each with the same early stopping as H2. `embed_dim = 128` and `learning_rate = 1e-3` stay at their baseline values.

In [11]:
DROPOUTS = [0.3, 0.5]
UNITS = [32, 64, 128]

results = []
for dropout in DROPOUTS:
    for units in UNITS:
        tf.keras.backend.clear_session()   # free the previous run's graph and GPU memory
        tf.random.set_seed(SEED)           # same starting weights for every run, so only the hyperparameters differ

        model = build_bilstm(units=units, dropout=dropout)
        early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
        history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10, batch_size=32, callbacks=[early_stop], verbose=0)

        best_epoch = int(np.argmin(history.history["val_loss"]))   # the epoch whose weights were restored
        val_pred = model.predict(X_val, verbose=0).argmax(axis=1)
        results.append({
            "dropout": dropout,
            "units": units,
            "epochs_run": len(history.history["loss"]),
            "best_epoch": best_epoch + 1,
            "train_acc": history.history["accuracy"][best_epoch],
            "val_acc": accuracy_score(y_val, val_pred),
            "val_macro_f1": f1_score(y_val, val_pred, average="macro"),
        })
        print("dropout={} units={} -> val macro-F1 {:.3f}".format(dropout, units, results[-1]["val_macro_f1"]))

tuning = pd.DataFrame(results).sort_values("val_macro_f1", ascending=False).reset_index(drop=True)
tuning.round(3)

dropout=0.3 units=32 -> val macro-F1 0.926
dropout=0.3 units=64 -> val macro-F1 0.935
dropout=0.3 units=128 -> val macro-F1 0.947
dropout=0.5 units=32 -> val macro-F1 0.951
dropout=0.5 units=64 -> val macro-F1 0.932
dropout=0.5 units=128 -> val macro-F1 0.931


,dropout,units,epochs_run,best_epoch,train_acc,val_acc,val_macro_f1
0,0.5,32,10,8,0.999,0.950,0.951
1,0.3,128,5,3,0.962,0.947,0.947
2,0.3,64,6,4,0.993,0.933,0.935
3,0.5,64,6,4,0.990,0.932,0.932
4,0.5,128,8,6,0.998,0.931,0.931
5,0.3,32,7,5,0.988,0.925,0.926


**What the tuning shows**: The best combination in the grid is dropout 0.5 with 32 units, at a val macro-F1 of 0.951, which is about the same as the baseline (0.952), so tuning did not improve on it. Neither hyperparameter shows a clear pattern: with dropout 0.3, more units scored higher (0.926, 0.935 and 0.947 for 32, 64 and 128 units), but with dropout 0.5, fewer units scored higher (0.951, 0.932 and 0.931). The winner's train/val gap is 0.049, about the same as the baseline's 0.047, so tuning did not narrow the gap either. The baseline setting (dropout 0.3, units 64) scored 0.952 in H2 but 0.935 in H3, so differences of up to about 0.02 can come from run-to-run variation alone. Unlike H2, every grid run starts from the same random seed, so the grid runs are compared on equal terms; dropout 0.5 with 32 units is therefore kept as the final setting, although its lead is within that variation.

#### H4. Final BiLSTM: Test Score

H3 picked dropout 0.5 with 32 units on val. The grid loop did not keep that model (each run replaced the previous one), so it is retrained here once with the same early stopping, and only then scored on test. This is the only time the BiLSTM sees the test split. The val score is printed first as a check: it can differ a little from H3's 0.951 because of run-to-run variation.

In [10]:
from sklearn.metrics import classification_report, confusion_matrix

BEST_DROPOUT, BEST_UNITS = 0.5, 32   # H3 winner on val macro-F1

tf.keras.backend.clear_session()
tf.random.set_seed(SEED)
bilstm = build_bilstm(units=BEST_UNITS, dropout=BEST_DROPOUT)
early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
history = bilstm.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10, batch_size=32, callbacks=[early_stop], verbose=0)

best_epoch = int(np.argmin(history.history["val_loss"]))
val_pred = bilstm.predict(X_val, verbose=0).argmax(axis=1)
print("Epochs run: {} | best epoch: {}".format(len(history.history["loss"]), best_epoch + 1))
print("Val accuracy: {:.3f} | Val macro-F1: {:.3f}".format(accuracy_score(y_val, val_pred), f1_score(y_val, val_pred, average="macro")))

# the single test evaluation
test_pred = bilstm.predict(X_test, verbose=0).argmax(axis=1)
print("Test accuracy: {:.3f} | Test macro-F1: {:.3f}\n".format(accuracy_score(y_test, test_pred), f1_score(y_test, test_pred, average="macro")))
print(classification_report(y_test, test_pred, target_names=LABELS, digits=3))

cm = pd.DataFrame(confusion_matrix(y_test, test_pred), index=LABELS, columns=LABELS)
cm.index.name, cm.columns.name = "true", "predicted"
cm

Epochs run: 7 | best epoch: 5
Val accuracy: 0.944 | Val macro-F1: 0.944
Test accuracy: 0.959 | Test macro-F1: 0.959

               precision    recall  f1-score   support

     business      0.952     0.967     0.959       183
entertainment      0.981     0.951     0.966       163
       health      0.921     0.974     0.947       192
       sports      0.974     0.979     0.977       192
   technology      0.972     0.920     0.945       187

     accuracy                          0.959       917
    macro avg      0.960     0.958     0.959       917
 weighted avg      0.959     0.959     0.959       917



predicted,business,entertainment,health,sports,technology
true,,,,,
business,177,0,3,1,2
entertainment,0,155,6,0,2
health,3,0,187,1,1
sports,2,0,2,188,0
technology,4,3,5,3,172


**What the test shows**: Test scores are a little higher than val: accuracy 0.959 against 0.944, and the same for macro-F1 (0.959 vs 0.944). The difference is within the run-to-run variation seen in H3, so the val score did not overestimate the model. `Sports` has the highest F1-score (0.977). The weakest labels are `technology` (F1 0.945) and `health` (F1 0.947), and they fail in different ways: `technology` misses articles (recall 0.920: 15 technology articles were predicted as other labels, 5 of them as `health`), while `health` collects wrong ones (precision 0.921: 16 articles from other labels were predicted as `health`, 6 of them from `entertainment`).

### I. IndoBERT

The second model is IndoBERT (`indobenchmark/indobert-base-p1`), a BERT model pre-trained on Indonesian text, fine-tuned here on `text_bert` (cleaned text that still has its stopwords, case and punctuation). Unlike the BiLSTM, it does not use our word vocabulary: its own tokenizer splits words into subwords, and the model reads at most 512 of them. Test stays untouched until the final IndoBERT run.

#### I1. Subword Length

IndoBERT's 512 limit counts subwords, not words, and two of those positions go to the special `[CLS]` and `[SEP]` tokens. This measures, on train only, how many subwords the tokenizer makes per word and how much of each article fits at different lengths, so the maximum length is chosen from evidence. Only the tokenizer is downloaded here, not the model.

In [17]:
from transformers import AutoTokenizer

BERT_NAME = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)

texts = train_df["text_bert"].tolist()
n_words = np.array([len(t.split()) for t in texts])
n_sub = np.array([len(ids) for ids in tokenizer(texts, add_special_tokens=False)["input_ids"]])

print("Lowercases the text: {}".format(tokenizer.do_lower_case))
print("Example: {}".format(tokenizer.tokenize(texts[0][:90])))
print("Subwords per word (all train): {:.2f}".format(n_sub.sum() / n_words.sum()))
print("Subwords per article: median {} | p95 {} | max {}".format(int(np.median(n_sub)), int(np.percentile(n_sub, 95)), n_sub.max()))

rows = []
for max_len in [128, 256, 384, 512]:
    room = max_len - 2   # [CLS] and [SEP] take two positions
    rows.append({
        "max_len": max_len,
        "articles_whole_%": round(100 * (n_sub <= room).mean(), 1),
        "subwords_kept_%": round(100 * np.minimum(n_sub, room).sum() / n_sub.sum(), 1),
    })
pd.DataFrame(rows)

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /indobenchmark/indobert-base-p1/resolve/main/tokenizer_config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000026DF62666D0>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: cd851053-53c9-46c3-aad9-8d45418e9ee4)')' thrown while requesting HEAD https://huggingface.co/indobenchmark/indobert-base-p1/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /indobenchmark/indobert-base-p1/resolve/main/tokenizer_config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000026DD9E50C40>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 5b32dcb4-727c-4c97-b7cf-1f471304c161)')' thrown while requesting HEAD https://huggingface

Lowercases the text: True
Example: ['harga', 'emas', 'antam', 'hari', 'ini', 'tidak', 'mengalami', 'perubahan', 'alias', 'stagn', '##an', 'usai', 'naik', 'tinggi', 'pada', 'ak']
Subwords per word (all train): 1.33
Subwords per article: median 420 | p95 820 | max 2186


,max_len,articles_whole_%,subwords_kept_%
0,128,5.2,26.9
1,256,12.6,53.4
2,384,40.8,74.5
3,512,69.3,87.1


**What the numbers show**: A maximum length of 512 subwords is still worth the extra cost, judging by how much more of each article the model can read: the share of articles that fit whole (`articles_whole_%`) and the share of subwords kept (`subwords_kept_%`). From 256 to 384, `articles_whole_%` grows by 28.2 points (12.6% to 40.8%) and `subwords_kept_%` by 21.1 points (53.4% to 74.5%). From 384 to 512, `articles_whole_%` rises from 40.8% to 69.3% and `subwords_kept_%` from 74.5% to 87.1%. We therefore choose `MAX_LEN_BERT = 512`, because the gain is still worth the complexity.

#### I2. Text to Subword Ids

Every split is turned into IndoBERT input with the tokenizer from I1: `input_ids` (the subword ids), `attention_mask` (1 for real subwords, 0 for padding) and `token_type_ids`. Articles longer than `MAX_LEN_BERT` keep their first 512 subwords and lose the rest; shorter ones are padded. The check counts how many articles were cut in each split.

In [18]:
def encode(texts):
    return tokenizer(texts, truncation=True, max_length=MAX_LEN_BERT, padding="max_length", return_tensors="np")

enc_train = encode(train_df["text_bert"].tolist())
enc_val = encode(val_df["text_bert"].tolist())
enc_test = encode(test_df["text_bert"].tolist())

for name, part, enc in [("train", train_df, enc_train), ("val", val_df, enc_val), ("test", test_df, enc_test)]:
    full_len = np.array([len(ids) for ids in tokenizer(part["text_bert"].tolist())["input_ids"]])   # with [CLS]/[SEP], untruncated
    print("{:5s} input_ids {} | cut to {}: {:.1f}% of articles".format(name, enc["input_ids"].shape, MAX_LEN_BERT, 100 * (full_len > MAX_LEN_BERT).mean()))

train input_ids (4277, 512) | cut to 512: 30.7% of articles
val   input_ids (916, 512) | cut to 512: 29.1% of articles
test  input_ids (917, 512) | cut to 512: 29.3% of articles


**What the numbers show**: All three splits are now encoded to the same length of 512 subwords: 4,277 train, 916 validation and 917 test articles. 30.7% of the train articles were longer than 512 subwords and were cut, which matches I1, where 69.3% fit whole. Validation (29.1%) and test (29.3%) are cut at almost the same rate, so the model is evaluated on the same kind of input it was trained on. The cut articles keep their first 512 subwords, which in news usually hold the title and the main point, so most of the useful information should remain.

#### I3. Baseline Fine-Tune

IndoBERT is fine-tuned once with starting settings before any tuning, to check that it learns. The pre-trained model gives one vector per article (`pooler_output`, built from the `[CLS]` position); a dropout layer and a 5-way softmax layer on top give the label. The whole model is trained, not only the new layer.

Two settings come from this laptop's GPU, not from tuning. TensorFlow gets about 3.4 GB of GPU memory, and at 512 subwords only a batch of 2 fits, with mixed precision (most of the maths in 16-bit numbers to save memory). Because a batch of 2 gives noisy updates, the learning rate starts low at 1e-5. Early stopping watches the val loss with patience 1 and keeps the best weights; BERT models usually need only 2 to 3 epochs, so the run is capped at 4. The score is on val, with accuracy and macro-F1 as for the BiLSTM. Test is not touched.

Note: the Hugging Face `TFAutoModelForSequenceClassification.fit` fails with this Keras version, so IndoBERT is wrapped in a plain Keras model instead.

In [19]:
from sklearn.metrics import accuracy_score, f1_score
from transformers import TFAutoModel

tf.keras.mixed_precision.set_global_policy("mixed_float16")   # 16-bit maths so 512 subwords fit in GPU memory
BERT_BATCH = 2   # largest batch that fits at 512 subwords on this GPU

def build_indobert(learning_rate=1e-5, dropout=0.1):
    bert = TFAutoModel.from_pretrained(BERT_NAME)
    ids = tf.keras.Input((MAX_LEN_BERT,), dtype=tf.int32, name="input_ids")
    mask = tf.keras.Input((MAX_LEN_BERT,), dtype=tf.int32, name="attention_mask")
    x = bert(input_ids=ids, attention_mask=mask).pooler_output
    x = tf.keras.layers.Dropout(dropout)(x)
    out = tf.keras.layers.Dense(len(LABELS), activation="softmax", dtype="float32")(x)   # float32 output keeps the loss stable
    model = tf.keras.Model([ids, mask], out)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def bert_inputs(enc):
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}

In [8]:
tf.keras.backend.clear_session()
tf.random.set_seed(SEED)
bert_model = build_indobert()
early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=1, restore_best_weights=True)
history = bert_model.fit(bert_inputs(enc_train), y_train, validation_data=(bert_inputs(enc_val), y_val), epochs=4, batch_size=BERT_BATCH, callbacks=[early_stop], verbose=2)

best_epoch = int(np.argmin(history.history["val_loss"]))
val_pred = bert_model.predict(bert_inputs(enc_val), batch_size=BERT_BATCH, verbose=0).argmax(axis=1)
print("Epochs run: {} | best epoch: {}".format(len(history.history["loss"]), best_epoch + 1))
print("Train accuracy at best epoch: {:.3f}".format(history.history["accuracy"][best_epoch]))
print("Val accuracy: {:.3f} | Val macro-F1: {:.3f}".format(accuracy_score(y_val, val_pred), f1_score(y_val, val_pred, average="macro")))

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some layers from the model checkpoint at indobenchmark/indobert-base-p1 were not used when initializing TFBertModel: ['nsp___cls', 'mlm___cls']
- This IS expected if you are initializing TFBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertModel were initialized from the model checkpoint at indobenchmark/indobert-base-p1.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for prediction

Epoch 1/4
2139/2139 - 443s - loss: 0.2952 - accuracy: 0.9095 - val_loss: 0.1808 - val_accuracy: 0.9432 - 443s/epoch - 207ms/step
Epoch 2/4
2139/2139 - 467s - loss: 0.1193 - accuracy: 0.9619 - val_loss: 0.1697 - val_accuracy: 0.9400 - 467s/epoch - 218ms/step
Epoch 3/4
2139/2139 - 372s - loss: 0.0653 - accuracy: 0.9797 - val_loss: 0.1754 - val_accuracy: 0.9498 - 372s/epoch - 174ms/step
Epochs run: 3 | best epoch: 2
Train accuracy at best epoch: 0.962
Val accuracy: 0.940 | Val macro-F1: 0.941


**What the baseline shows**: IndoBERT learns the task quickly: after one epoch the val accuracy is already 0.943, because the model was pre-trained on Indonesian text. Early stopping keeps epoch 2, where the val loss is lowest (0.170), and stops after epoch 3 when the val loss rises again. At epoch 2 the train accuracy is 0.962 and the val accuracy is 0.940 (macro-F1 0.941). That is a gap of about 0.02, smaller than the BiLSTM baseline's 0.05, so IndoBERT overfits less. Its val score is slightly below the BiLSTM baseline's (macro-F1 0.952), a difference within the BiLSTM's own run-to-run variation.

#### I4. Tuning: Learning Rate, Then Dropout

Two hyperparameters are tuned, one at a time, because each IndoBERT run takes about 20 minutes on this laptop. Batch size and length stay fixed (the GPU only fits a batch of 2 at 512 subwords).

1. **Learning rate**: 2e-5 is compared with the baseline's 1e-5, both with dropout 0.1. A larger rate learns faster but can overshoot.
2. **Dropout** on the new layer: 0.3 is tried with whichever learning rate won step 1, against 0.1.

The I3 run is reused as the 1e-5 / 0.1 result instead of being trained again. Changing one setting at a time does not test every combination, so it can miss a pair that only works together. Every run uses the same early stopping as I3 and is scored on val by macro-F1. Test is not touched.

In [9]:
import gc

def run_indobert(learning_rate, dropout):
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    model = build_indobert(learning_rate=learning_rate, dropout=dropout)
    early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=1, restore_best_weights=True)
    hist = model.fit(bert_inputs(enc_train), y_train, validation_data=(bert_inputs(enc_val), y_val), epochs=4, batch_size=BERT_BATCH, callbacks=[early_stop], verbose=2)
    best = int(np.argmin(hist.history["val_loss"]))
    pred = model.predict(bert_inputs(enc_val), batch_size=BERT_BATCH, verbose=0).argmax(axis=1)
    del model
    gc.collect()
    return {"learning_rate": learning_rate, "dropout": dropout, "epochs_run": len(hist.history["loss"]), "best_epoch": best + 1,
            "train_acc": round(hist.history["accuracy"][best], 3), "val_acc": round(accuracy_score(y_val, pred), 3),
            "val_macro_f1": round(f1_score(y_val, pred, average="macro"), 3)}

bert_model = None   # frees the I3 model's GPU memory; the final model is retrained in I5
gc.collect()

# I3 baseline, copied from its printed output so it is not trained again
bert_results = [{"learning_rate": 1e-5, "dropout": 0.1, "epochs_run": 3, "best_epoch": 2, "train_acc": 0.962, "val_acc": 0.940, "val_macro_f1": 0.941}]
pd.DataFrame(bert_results)

,learning_rate,dropout,epochs_run,best_epoch,train_acc,val_acc,val_macro_f1
0,0.00001,0.1,3,2,0.962,0.94,0.941


In [10]:
# step 1: learning rate (about 20 min)
bert_results = bert_results[:1] + [run_indobert(learning_rate=2e-5, dropout=0.1)]
pd.DataFrame(bert_results)

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some layers from the model checkpoint at indobenchmark/indobert-base-p1 were not used when initializing TFBertModel: ['mlm___cls', 'nsp___cls']
- This IS expected if you are initializing TFBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertModel were initialized from the model checkpoint at indobenchmark/indobert-base-p1.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for prediction

Epoch 1/4
2139/2139 - 370s - loss: 0.3228 - accuracy: 0.8992 - val_loss: 0.1895 - val_accuracy: 0.9400 - 370s/epoch - 173ms/step
Epoch 2/4
2139/2139 - 336s - loss: 0.1673 - accuracy: 0.9516 - val_loss: 0.2631 - val_accuracy: 0.9203 - 336s/epoch - 157ms/step


,learning_rate,dropout,epochs_run,best_epoch,train_acc,val_acc,val_macro_f1
0,0.00001,0.1,3,2,0.962,0.94,0.941
1,0.00002,0.1,2,1,0.899,0.94,0.939


In [11]:
# step 2: dropout, at the learning rate that won step 1 (about 20 min)
best_lr = max(bert_results[:2], key=lambda r: r["val_macro_f1"])["learning_rate"]
print("Learning rate kept from step 1: {}".format(best_lr))
bert_results = bert_results[:2] + [run_indobert(learning_rate=best_lr, dropout=0.3)]
pd.DataFrame(bert_results).sort_values("val_macro_f1", ascending=False)

'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: fa76933d-d13c-44d4-a640-8f53b0a92add)')' thrown while requesting HEAD https://huggingface.co/indobenchmark/indobert-base-p1/resolve/main/config.json
Retrying in 1s [Retry 1/5].


Learning rate kept from step 1: 1e-05


Some layers from the model checkpoint at indobenchmark/indobert-base-p1 were not used when initializing TFBertModel: ['mlm___cls', 'nsp___cls']
- This IS expected if you are initializing TFBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertModel were initialized from the model checkpoint at indobenchmark/indobert-base-p1.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions without further training.


Epoch 1/4
2139/2139 - 354s - loss: 0.3027 - accuracy: 0.9011 - val_loss: 0.1837 - val_accuracy: 0.9334 - 354s/epoch - 165ms/step
Epoch 2/4
2139/2139 - 335s - loss: 0.1265 - accuracy: 0.9600 - val_loss: 0.1665 - val_accuracy: 0.9465 - 335s/epoch - 157ms/step
Epoch 3/4
2139/2139 - 336s - loss: 0.0869 - accuracy: 0.9722 - val_loss: 0.1541 - val_accuracy: 0.9552 - 336s/epoch - 157ms/step
Epoch 4/4
2139/2139 - 333s - loss: 0.0543 - accuracy: 0.9839 - val_loss: 0.1466 - val_accuracy: 0.9607 - 333s/epoch - 156ms/step


,learning_rate,dropout,epochs_run,best_epoch,train_acc,val_acc,val_macro_f1
2,0.00001,0.3,4,4,0.984,0.961,0.961
0,0.00001,0.1,3,2,0.962,0.940,0.941
1,0.00002,0.1,2,1,0.899,0.940,0.939


**What the tuning shows**: From the results, we can see that learning rate 2e-5 was too fast: after the first epoch the val loss jumped from 0.190 to 0.263, so early stopping kept epoch 1. Its macro-F1 (0.939) is almost the same as 1e-5 (0.941), and since 1e-5 was slightly better, step 2 kept 1e-5. Dropout 0.3 was the clear winner, with val macro-F1 0.961, 0.020 above the baseline. With dropout 0.1 the val loss started rising after epoch 2, but with dropout 0.3 it kept falling every epoch (0.184, 0.167, 0.154, 0.147). The stronger dropout slowed overfitting, so the model could train longer and keep improving. The gap between train and val accuracy stayed small (0.984 vs 0.961, about 0.02), the same as the baseline. There are two limits. The 0.3 run was still improving when it reached the 4-epoch cap, so more epochs might score higher. And a gain of 0.020 is about the size of the run-to-run variation seen with the BiLSTM, so the steady fall in val loss is the stronger evidence that dropout 0.3 helps. The final IndoBERT therefore uses learning rate 1e-5 and dropout 0.3.

#### I5. Final IndoBERT: Test Score

I4 picked learning rate 1e-5 with dropout 0.3 on val. The tuning helper did not keep that model, so it is retrained here once with the same early stopping and the same 4-epoch cap, and only then scored on test. This is the only time IndoBERT sees the test split. The val score is printed first as a check: it can differ a little from I4's 0.961 because of run-to-run variation.

In [20]:
import gc
from sklearn.metrics import classification_report, confusion_matrix

BEST_BERT_LR, BEST_BERT_DROPOUT = 1e-5, 0.3   # I4 winner on val macro-F1

bert_model = None   # frees any earlier IndoBERT model's GPU memory
gc.collect()
tf.keras.backend.clear_session()
tf.random.set_seed(SEED)
bert_model = build_indobert(learning_rate=BEST_BERT_LR, dropout=BEST_BERT_DROPOUT)
early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=1, restore_best_weights=True)
history = bert_model.fit(bert_inputs(enc_train), y_train, validation_data=(bert_inputs(enc_val), y_val), epochs=4, batch_size=BERT_BATCH, callbacks=[early_stop], verbose=2)

best_epoch = int(np.argmin(history.history["val_loss"]))
val_pred = bert_model.predict(bert_inputs(enc_val), batch_size=BERT_BATCH, verbose=0).argmax(axis=1)
print("Epochs run: {} | best epoch: {}".format(len(history.history["loss"]), best_epoch + 1))
print("Val accuracy: {:.3f} | Val macro-F1: {:.3f}".format(accuracy_score(y_val, val_pred), f1_score(y_val, val_pred, average="macro")))

# the single test evaluation
test_pred = bert_model.predict(bert_inputs(enc_test), batch_size=BERT_BATCH, verbose=0).argmax(axis=1)
print("Test accuracy: {:.3f} | Test macro-F1: {:.3f}\n".format(accuracy_score(y_test, test_pred), f1_score(y_test, test_pred, average="macro")))
print(classification_report(y_test, test_pred, target_names=LABELS, digits=3))

cm = pd.DataFrame(confusion_matrix(y_test, test_pred), index=LABELS, columns=LABELS)
cm.index.name, cm.columns.name = "true", "predicted"
cm

Some layers from the model checkpoint at indobenchmark/indobert-base-p1 were not used when initializing TFBertModel: ['mlm___cls', 'nsp___cls']
- This IS expected if you are initializing TFBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertModel were initialized from the model checkpoint at indobenchmark/indobert-base-p1.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions without further training.


Epoch 1/4
2139/2139 - 562s - loss: 0.3195 - accuracy: 0.8990 - val_loss: 0.2000 - val_accuracy: 0.9389 - 562s/epoch - 263ms/step
Epoch 2/4
2139/2139 - 501s - loss: 0.1233 - accuracy: 0.9628 - val_loss: 0.1947 - val_accuracy: 0.9400 - 501s/epoch - 234ms/step
Epoch 3/4
2139/2139 - 385s - loss: 0.0824 - accuracy: 0.9731 - val_loss: 0.1480 - val_accuracy: 0.9596 - 385s/epoch - 180ms/step
Epoch 4/4
2139/2139 - 366s - loss: 0.0659 - accuracy: 0.9794 - val_loss: 0.1964 - val_accuracy: 0.9487 - 366s/epoch - 171ms/step
Epochs run: 4 | best epoch: 3
Val accuracy: 0.960 | Val macro-F1: 0.960
Test accuracy: 0.963 | Test macro-F1: 0.964

               precision    recall  f1-score   support

     business      0.962     0.967     0.965       183
entertainment      1.000     0.963     0.981       163
       health      0.953     0.943     0.948       192
       sports      0.995     0.974     0.984       192
   technology      0.914     0.968     0.940       187

     accuracy                      

predicted,business,entertainment,health,sports,technology
true,,,,,
business,177,0,2,0,4
entertainment,0,157,2,0,4
health,4,0,181,0,7
sports,0,0,3,187,2
technology,3,0,2,1,181


**What the test shows**: On test, IndoBERT scores 0.963 accuracy and 0.964 macro-F1, only slightly above the BiLSTM (0.959 and 0.959). The difference of about 0.005 is smaller than the run-to-run variation seen in H3, so on these numbers the two models are close to equal. IndoBERT is better on four labels: business 0.965 (BiLSTM 0.959), entertainment 0.981 (0.966), health 0.948 (0.947) and sports 0.984 (0.977), and slightly worse on technology, 0.940 against 0.945. The val check (0.960) is close to I4's 0.961, so the retrain behaved like the tuning run; this time the val loss was lowest at epoch 3 and rose at epoch 4, so early stopping kept epoch 3. Technology is IndoBERT's weakest label: its precision is 0.914, because 17 articles from other labels were predicted as technology, mostly health articles (7). Entertainment has a precision of 1.000, so no other article was predicted as entertainment. The remaining mistakes are spread thinly, with no pair of labels confused more than 7 times.